# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

Ваши ответы и объяснения:

1. `[1. 2. 3.] [0 1 1]` — `/` всегда даёт float, `//` целочисленное деление: 2//4=0, 4//4=1, 6//4=1.
2. `[4. 4. 4.]` — `axis=1` суммирует по столбцам внутри каждой из 3 строк, в строке 4 единицы.
3. `[[0 1 2] [1 2 3] [2 3 4]]` — broadcasting: строка (3,) и столбец (3,1) растягиваются до (3,3).
4. `6.0` — корни 1, 2, 3, их сумма 6, `sqrt` возвращает float.
5. `7.5` — квадраты 1, 4, 9, 16, сумма 30, среднее 30/4.

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [2]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [3]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [4]:
months = np.array(["2026-09", "2026-10"])
rev = np.array([amount[clean & (month == m)].sum() for m in months])
change = (rev[1] - rev[0]) / rev[0] * 100
print("строк в clean:", clean.sum())
print(f"сентябрь: {rev[0]:,.0f} ₸, октябрь: {rev[1]:,.0f} ₸, изменение: {change:+.1f}%")
print(f"Выручка в октябре изменилась на {change:+.1f}% относительно сентября.")

строк в clean: 4739
сентябрь: 35,243,410 ₸, октябрь: 34,419,520 ₸, изменение: -2.3%
Выручка в октябре изменилась на -2.3% относительно сентября.


## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [5]:
cats = np.unique(cat)
cat_table = np.array([[amount[clean & (cat == c) & (month == m)].sum() for m in months] for c in cats])
for c, r in zip(cats, cat_table):
    print(f"{c:<12} {r[0]:>12,.0f} {r[1]:>12,.0f}")
cat_total = cat_table.sum(axis=1)
cat_change = (cat_table[:, 1] - cat_table[:, 0]) / cat_table[:, 0] * 100
print(f"Больше всего выручки за два месяца: {cats[cat_total.argmax()]} ({cat_total.max():,.0f} ₸)")
print(f"Сильнее всех упала: {cats[cat_change.argmin()]} ({cat_change.min():+.1f}%)")

Дом             4,997,730    5,125,790
Книги           1,065,300    1,140,910
Красота         2,610,950    2,678,990
Одежда          9,167,100    8,505,730
Электроника    17,402,330   16,968,100
Больше всего выручки за два месяца: Электроника (34,370,430 ₸)
Сильнее всех упала: Одежда (-7.2%)


## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [6]:
cat_count = np.array([[(clean & (cat == c) & (month == m)).sum() for m in months] for c in cats])
avg_check = cat_table / cat_count
for c, n, a in zip(cats, cat_count, avg_check):
    print(f"{c:<12} {n[1]:>5} {a[1]:>10,.0f}")
print(f"Самый высокий средний чек в октябре: {cats[avg_check[:, 1].argmax()]} ({avg_check[:, 1].max():,.0f} ₸)")

Дом            476     10,768
Книги          299      3,816
Красота        435      6,159
Одежда         652     13,046
Электроника    505     33,600
Самый высокий средний чек в октябре: Электроника (33,600 ₸)


## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [7]:
cities = np.unique(city[clean])
table = np.array([[amount[clean & (city == c) & (month == m)].sum() for m in months] for c in cities])
city_share = table / table.sum(axis=0)
for c, r in zip(cities, city_share):
    print(f"{c:<10} {r[0]:>7.1%} {r[1]:>7.1%}")
print("суммы столбцов:", city_share.sum(axis=0))
s = city_share[cities == "Шымкент"][0]
print(f"Доля Шымкента: {s[0]:.1%} -> {s[1]:.1%} ({(s[1] - s[0]) * 100:+.1f} п.п.)")

Актобе        7.3%    7.4%
Алматы       41.1%   42.5%
Астана       25.6%   25.7%
Атырау        2.6%    2.8%
Караганда    10.4%    9.8%
Шымкент      12.9%   11.7%
суммы столбцов: [1. 1.]
Доля Шымкента: 12.9% -> 11.7% (-1.2 п.п.)


## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [8]:
city_diff = table[:, 1] - table[:, 0]
contrib = city_diff / city_diff.sum()
for c, d, k in zip(cities, city_diff, contrib):
    print(f"{c:<10} {d:>12,.0f} {k:>7.1%}")
print(f"Шымкент даёт {contrib[cities == 'Шымкент'][0]:.1%} падения выручки.")
print(f"Причину падения ищите в первую очередь в городе {cities[contrib.argmax()]}: он даёт крупнейший вклад.")

Актобе          -36,540    4.4%
Алматы          141,950  -17.2%
Астана         -183,690   22.3%
Атырау           45,870   -5.6%
Караганда      -276,840   33.6%
Шымкент        -514,640   62.5%
Шымкент даёт 62.5% падения выручки.
Причину падения ищите в первую очередь в городе Шымкент: он даёт крупнейший вклад.


## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [9]:
print(table.shape, table.sum(axis=1).shape)
share = table / table.sum(axis=1, keepdims=True)
print(share)
print(share.sum(axis=1))

(6, 2) (6,)
[[0.50357019 0.49642981]
 [0.49756469 0.50243531]
 [0.50513538 0.49486462]
 [0.48788975 0.51211025]
 [0.51971739 0.48028261]
 [0.52991168 0.47008832]]
[1. 1. 1. 1. 1. 1.]


`table` имеет форму (6, 2), а `table.sum(axis=1)` — форму (6,). Broadcasting выравнивает формы справа: 2 и 6 не совпадают, поэтому ошибка. Сумму нужно оставить столбцом (6, 1) через `keepdims=True`.

## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [10]:
x = amount[clean]
z = (x - x.mean()) / x.std()
out = np.abs(z) > 3
print(f"|z| > 3: {out.sum()} заказов, доля {out.mean():.2%}")
print(f"mean(z) = {z.mean():.2e}, std(z) = {z.std():.3f}")
print(f"{out.sum()} заказов ({out.mean():.2%}) аномально крупные или мелкие, их стоит проверить отдельно.")

|z| > 3: 87 заказов, доля 1.84%
mean(z) = 7.05e-17, std(z) = 1.000
87 заказов (1.84%) аномально крупные или мелкие, их стоит проверить отдельно.


## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [11]:
ok = clean & ~np.isnan(days)
X = np.column_stack([amount[ok], days[ok]])
X_scaled = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))
print(X.shape)
print("min:", X_scaled.min(axis=0), "max:", X_scaled.max(axis=0))

(4595, 2)
min: [0. 0.] max: [1. 1.]


Сумма заказа измеряется в тысячах тенге, а срок доставки — в единицах дней. Без масштабирования модели, которые считают расстояния или используют градиентный спуск, будут почти полностью ориентироваться на сумму, а дни станут незаметными. Приведение к [0, 1] даёт признакам равный вес и ускоряет обучение.